# BanglaKnight: prepare_submission_file

Turns your predictions into the file Codabench accepts, and checks it first:

- one row for every `id` of the phase's blind file, no extra or repeated ids,
- labels 0, 1, 2 or 3 (label names such as `Harmful` are converted),
- written as `predictions.csv` (`id,label`) inside `submission.zip`.

Upload `submission.zip` under **My Submissions** on Codabench, in the right phase.

## 1. Your files

- `BLIND_FILE`: the blind input file of the phase (`banglaknight_devtest.csv` in the Development phase, `banglaknight_test.csv` in the Test phase). It is downloaded from the task website if it is not in the working folder.
- `PRED_FILE`: your predictions as a CSV with an `id` column and a `label` column (for example `my_predictions.csv` from the starter notebook).

In [ ]:
import os, urllib.request, zipfile
import pandas as pd

BLIND_FILE = "banglaknight_devtest.csv"   # Test phase: "banglaknight_test.csv"
PRED_FILE = "my_predictions.csv"
OUT_ZIP = "submission.zip"

if not os.path.exists(BLIND_FILE):
    urllib.request.urlretrieve(f"https://banglaknight.github.io/data/files/{BLIND_FILE}", BLIND_FILE)

## 2. Check

In [ ]:
NAMES = {"safe": 0, "offensive": 1, "harmful": 2, "severe": 3}

def to_label(v):
    s = str(v).strip().lower()
    if s in NAMES:
        return NAMES[s]
    f = float(s)
    if f != int(f) or int(f) not in (0, 1, 2, 3):
        raise ValueError(v)
    return int(f)

blind = pd.read_csv(BLIND_FILE)
pred = pd.read_csv(PRED_FILE)
pred.columns = [c.strip().lower() for c in pred.columns]
problems = []
if not {"id", "label"} <= set(pred.columns):
    problems.append(f"{PRED_FILE} needs the columns id and label; found {list(pred.columns)}")
else:
    pred["id"] = pred["id"].astype(str).str.strip()
    bad = []
    for v in pred["label"]:
        try:
            to_label(v)
        except ValueError:
            bad.append(v)
    if bad:
        problems.append(f"{len(bad)} labels are not 0-3 or a label name, e.g. {bad[:5]}")
    dups = pred["id"][pred["id"].duplicated()].tolist()
    if dups:
        problems.append(f"{len(dups)} repeated ids, e.g. {dups[:5]}")
    need, have = set(blind["id"]), set(pred["id"])
    if need - have:
        problems.append(f"{len(need - have)} ids of {BLIND_FILE} have no prediction, e.g. {sorted(need - have)[:5]}")
    if have - need:
        problems.append(f"{len(have - need)} ids are not in {BLIND_FILE}, e.g. {sorted(have - need)[:5]}")

if problems:
    print("NOT READY:")
    for p in problems:
        print(" -", p)
else:
    print(f"OK: {len(pred)} predictions, one for every row of {BLIND_FILE}")

## 3. How consistent are your three forms?

Not required, but a good system gives the same label to all three forms of a prompt.

In [ ]:
if not problems:
    m = blind[["id", "prompt_id"]].merge(pred.assign(label=pred["label"].map(to_label)), on="id")
    agree = (m.groupby("prompt_id")["label"].nunique() == 1).mean()
    print(f"prompts with the same label in all three forms: {agree:.1%}")
    print(m["label"].value_counts().sort_index().rename({0: "Safe", 1: "Offensive", 2: "Harmful", 3: "Severe"}).to_string())

## 4. Write `submission.zip`

In [ ]:
if problems:
    raise SystemExit("Fix the problems above first.")
out = blind[["id"]].merge(pred.assign(label=pred["label"].map(to_label))[["id", "label"]], on="id", how="left")
with zipfile.ZipFile(OUT_ZIP, "w", zipfile.ZIP_DEFLATED) as z:
    z.writestr("predictions.csv", out.to_csv(index=False))
print(f"wrote {OUT_ZIP} with predictions.csv ({len(out)} rows). Upload it on Codabench.")
try:
    from google.colab import files   # in Colab, download it
    files.download(OUT_ZIP)
except ImportError:
    pass